# 00b — Compare Drive Audits

Loads two JSON reports produced by `00_data_audit.ipynb` (one per Google account)
and prints a side-by-side comparison.

**Run after:** `00_data_audit.ipynb` on **both** Google accounts.

**Inputs:** Two JSON files — `audit_report_account1.json` and `audit_report_account2.json`  
**Output:** Side-by-side folder comparison, deduplication analysis, and a recommendation
on which account's data to use for training.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install tabulate -q

In [ ]:
# ── Cell 2: Imports & Drive mount ─────────────────────────────────────────────
import json
from pathlib import Path
import numpy as np
import pandas as pd
from tabulate import tabulate

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 3: ✏️  CONFIGURE THIS ─────────────────────────────────────────────────
# Paths to the two JSON audit reports.
# Account 2's JSON must be shared to this Drive first:
#   On Account 2 Drive → right-click JSON → Share → copy link
#   On Account 1 Drive → 'Shared with me' → right-click → 'Add shortcut to My Drive'

REPORT_ACCOUNT1 = '/content/drive/MyDrive/GlioGrade/audit_report_account1.json'
REPORT_ACCOUNT2 = '/content/drive/MyDrive/GlioGrade/audit_report_account2.json'

# Labels shown in output
LABEL_A = 'Account 1'
LABEL_B = 'Account 2'

print('Config set.')

In [ ]:
# ── Cell 4: Load reports ───────────────────────────────────────────────────────

def load_report(path, label):
    p = Path(path)
    if not p.exists():
        print(f'  ❌  {label}: file not found at {path}')
        return None
    with open(p) as f:
        data = json.load(f)
    print(f'  ✅  {label}: loaded  (generated {data["generated_at"]})')
    print(f'       Folders scanned: {len(data["folders_scanned"])}')
    return data

print('Loading reports...')
report_a = load_report(REPORT_ACCOUNT1, LABEL_A)
report_b = load_report(REPORT_ACCOUNT2, LABEL_B)

if report_a is None or report_b is None:
    raise RuntimeError('One or both reports missing. Run 00_data_audit.ipynb on both accounts first.')

In [ ]:
# ── Cell 5: Side-by-side folder comparison ────────────────────────────────────

SEP  = '═' * 80
SEP2 = '─' * 80

def folder_summary_rows(report, label):
    rows = []
    for fpath, r in report['folder_reports'].items():
        folder_name = Path(fpath).name
        if not r.get('exists'):
            rows.append([label, folder_name, '❌ not found', '', '', '', ''])
            continue
        good = [s for s in r.get('samples', []) if s.get('ok')]
        avg_zero = np.mean([s['zero_frac'] for s in good]) if good else None
        stripped = ('✅' if avg_zero > 0.3 else '⚠️') if avg_zero is not None else '?'
        split    = '✅' if r.get('split_detected') else '⚠️'
        shapes   = list({str(s['shape']) for s in good if s.get('ok')})
        types    = ', '.join(sorted(r.get('scan_types', {}).keys())[:4])
        rows.append([
            label,
            folder_name,
            r.get('n_nifti', 0),
            r.get('n_patients', 0),
            stripped,
            split,
            types or '?',
        ])
    return rows

rows_a = folder_summary_rows(report_a, LABEL_A)
rows_b = folder_summary_rows(report_b, LABEL_B)

headers = ['Account', 'Folder', 'NIfTI files', 'Patients', 'Skull-stripped', 'Has split', 'Scan types']

print(SEP)
print('  FOLDER COMPARISON')
print(SEP)
print(tabulate(rows_a + rows_b, headers=headers, tablefmt='simple'))
print(f'\n{SEP}')

In [ ]:
# ── Cell 6: Patient ID overlap analysis ───────────────────────────────────────

def all_ids(report):
    ids = set()
    for r in report['folder_reports'].values():
        if r.get('exists'):
            ids.update(r.get('patient_ids', []))
    return ids

ids_a = all_ids(report_a)
ids_b = all_ids(report_b)
overlap   = ids_a & ids_b
only_in_a = ids_a - ids_b
only_in_b = ids_b - ids_a
union     = ids_a | ids_b

print(SEP)
print('  PATIENT ID OVERLAP')
print(SEP)
print(f'  {LABEL_A} total unique patient IDs:  {len(ids_a)}')
print(f'  {LABEL_B} total unique patient IDs:  {len(ids_b)}')
print(f'  Union (combined unique):             {len(union)}')
print(f'  Overlap (same patients both):        {len(overlap)}')
print(f'  Only in {LABEL_A}:                   {len(only_in_a)}')
print(f'  Only in {LABEL_B}:                   {len(only_in_b)}')

if overlap:
    print(f'\n  ⚠️  Overlap sample (first 10): {sorted(overlap)[:10]}')
    print('     These patients appear in BOTH accounts — if used together,')
    print('     deduplicate by patient ID to avoid data leakage between splits.')
else:
    print('\n  ✅  No patient ID overlap — both accounts have fully disjoint sets.')

print(f'\n{SEP}')

In [ ]:
# ── Cell 7: Preprocessing state comparison ────────────────────────────────────
# Shows per-folder normalization and skull-stripping state for each account.

def preproc_rows(report, label):
    rows = []
    for fpath, r in report['folder_reports'].items():
        if not r.get('exists'):
            continue
        good = [s for s in r.get('samples', []) if s.get('ok')]
        if not good:
            continue
        norms  = [s.get('normalization', '?') for s in good]
        shapes = list({str(s['shape']) for s in good})
        voxels = list({str(s['voxel_mm']) for s in good})
        avg_zero = np.mean([s['zero_frac'] for s in good])
        rows.append([
            label,
            Path(fpath).name,
            norms[0] if len(set(norms)) == 1 else 'mixed: ' + str(set(norms)),
            '✅' if avg_zero > 0.3 else '⚠️ no',
            shapes[0] if len(shapes) == 1 else 'mixed',
            voxels[0] if len(voxels) == 1 else 'mixed',
        ])
    return rows

headers2 = ['Account', 'Folder', 'Normalization', 'Skull-stripped', 'Volume shape', 'Voxel spacing']
rows2 = preproc_rows(report_a, LABEL_A) + preproc_rows(report_b, LABEL_B)

print(SEP)
print('  PREPROCESSING STATE COMPARISON')
print(SEP)
if rows2:
    print(tabulate(rows2, headers=headers2, tablefmt='simple'))
else:
    print('  No sample data available (check that SAMPLE_SIZE > 0 in 00_data_audit.ipynb).')
print(f'\n{SEP}')

In [ ]:
# ── Cell 8: Recommendation ─────────────────────────────────────────────────────
# Scores each folder across both accounts and prints a ranked recommendation.

def score_folder(r):
    """Simple heuristic score: higher = better candidate for training."""
    if not r.get('exists'):
        return -1
    score = 0
    score += r.get('n_patients', 0)          # more patients = better
    if r.get('split_detected'):  score += 20  # already split saves work
    good = [s for s in r.get('samples', []) if s.get('ok')]
    if good:
        avg_zero = np.mean([s['zero_frac'] for s in good])
        if avg_zero > 0.3:      score += 30   # skull-stripped
        norms = [s.get('normalization', '') for s in good]
        if any('z-score' in n for n in norms):   score += 20
        elif any('min-max' in n for n in norms): score += 10
    return score

candidates = []
for label, report in [(LABEL_A, report_a), (LABEL_B, report_b)]:
    for fpath, r in report['folder_reports'].items():
        candidates.append({
            'account': label,
            'folder':  Path(fpath).name,
            'path':    fpath,
            'score':   score_folder(r),
            'n_patients': r.get('n_patients', 0),
            'split':   '✅' if r.get('split_detected') else '⚠️',
        })

candidates.sort(key=lambda x: -x['score'])

print(SEP)
print('  RECOMMENDATION  (higher score = better candidate for training)')
print(SEP)
rec_rows = [
    [c['account'], c['folder'], c['n_patients'], c['split'], c['score']]
    for c in candidates
]
print(tabulate(rec_rows, headers=['Account', 'Folder', 'Patients', 'Split', 'Score'], tablefmt='simple'))

best = candidates[0]
print(f'\n  🏆  Best candidate:  [{best["account"]}]  {best["folder"]}')
print(f'      Full path:       {best["path"]}')
print(f'      Score:           {best["score"]}  (patients: {best["n_patients"]}, split: {best["split"]})')
print()
print('  Scoring rubric:  +1 per patient, +20 if split exists, +30 skull-stripped, +20 z-score, +10 min-max')
print('  ⚠️  Always verify manually — score is a heuristic, not a guarantee.')
print(f'\n{SEP}')